# 03 - CatBoost on lemmatized vs. raw text with keyword features

In [ ]:
DATA_DIR = "../data"

In [ ]:
!pip install -q CatBoost
!pip install -q datasets

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    # text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE\s+(\w+)\]', r'\1', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9% ]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
        "org": "orang",
        "pemda": "pemerintah daerah",
        "pemkot": "pemerintah kota",
        "pemprov": "pemerintah provinsi",
        "pemkab": "pemerintah kabupaten",
        "pemilu": "pemilihan umum",
        "pilkada": "pemilihan kepala daerah",
        "pilpres": "pemilihan presiden",
        "pilgub": "pemilihan gubernur",
        "pilbup": "pemilihan bupati",
        "pilwali": "pemilihan wali kota",
        "pilleg": "pemilihan legislatif",
        "pilpres": "pemilihan presiden",
        "ajg" : "bajingan",
        "anjg" : "banjingan",
        "anjir" : "banjingan",
        "anjay" : "banjingan",
        "anjayy" : "banjingan",
        "anjayyy" : "banjingan",
        "ikn" : "ibu kota negara",
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")
    text = re.sub("alam ganjar", 'anaknya ganjar', text, flags=re.IGNORECASE)
    text = re.sub("%", 'persen', text, flags=re.IGNORECASE)


    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

In [ ]:
train_fix_df = pd.read_csv(f'{DATA_DIR}/train_fix.csv')

# get index that is true in is_false column
false_idx = train_fix_df['is_false'] == 1

In [ ]:
train_fix_df[false_idx]

                                                   text             label  \
19    RT Pura pura buat dukung anies, padahal buat n...         Demografi   
26    RT Anies Janji Lagi Siap Koreksi UU Cipta Kerj...         Demografi   
90    RT Testimoni Pendeta dari Magetan yang datang ...           Ekonomi   
146   @naf+qRZFlR2YaY0+kjZ8iEYBXa/MLw53GV3MNjee6hs= ...           Ekonomi   
165   RT Gw gak habis pikir liat orang bisa pilih An...           Ekonomi   
...                                                 ...               ...   
4982  @YTkWLimSUm+nkwJUPkFHnBHpbfV5foLz1CxHm3Ku2QE= ...  Sumber Daya Alam   
4987  @zcQsV8O6QuyJZLzWAJB9ZHpmMqYiUfzWJvmCs9k1rGo= ...  Sumber Daya Alam   
4993  RT Gw bilang juga apa, terasa juga Nasdem sama...  Sumber Daya Alam   
4994  RT @zybpLwj2I9C9dDkdD80OzHaOl/Tsbuh10bLxEkc5lA...  Sumber Daya Alam   
4999  Acara ini tidak hanya memasak, tetapi Calon Pr...  Sumber Daya Alam   

      is_false correct_label Unnamed: 4  Unnamed: 5  
19        True       

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/bdc_train.csv', sep=";")
test_df = pd.read_csv(f'{DATA_DIR}/bdc_test.csv', sep=";")
train_df.head()
# remove train_df based on false index
train_df = train_df[~false_idx]

<ipython-input-5-a5106e766f5f>:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  train_df = train_df[~false_idx]


In [ ]:
train_df

                                                   text  \
0     Kunjungan Prabowo ini untuk meresmikan dan men...   
1     RT Anies dapat tepuk tangan meriah saat jadi R...   
2     @CIqXqwGAT04tMtx4OCATxjoVq7vv/Y8HeYaIOgMFg8Y= ...   
3     RT @L3R8XFBw3WGbxRPSj0/0hHZTbqVGX7qtfwRg9zmhK7...   
4     Anies Baswedan Harap ASN termasuk TNI dan Polr...   
...                                                 ...   
4992  Para Kiyai berpandangan bahwa pasangan Ganjar-...   
4995  RT @l1DsGGe2xObT3t72dHwqlT58X7jvEYtEnauZIZSYwS...   
4996  Masyarakat yakin bahwa Prabowo-Gibran memiliki...   
4997  imo both are irrational but yg satu jauh lebih...   
4998  @cIIGSdjaPrmAQY1E4gWnLpIZSdyQn8ZMhjJzgOsxfRM= ...   

                        label  
0            Sumber Daya Alam  
1                     Politik  
2                   Demografi  
3                     Politik  
4                     Politik  
...                       ...  
4992                  Politik  
4995                  Politik  
499

In [ ]:
test_df

,IDText,Text
0,TXT0001,Lu mau org2 pro-demokrasi di negara ini bisa p...
1,TXT0002,Prabowo ditanya soal hutang luar negeri dia me...
2,TXT0003,kiki_daliyo Ganjar Pranowo itulah beliau soso...
3,TXT0004,@kumparan Prabowo Gibran yang bisa melakukan i...
4,TXT0005,@sniperruben45 @uda_zulhendra @ainunnajib Lah ...
...,...,...
995,TXT0996,"Bikin bangga deh, Ganjar-Mahfud mau alokasikan..."
996,TXT0997,Pak Jokowi sebelum pilpres 2024 berbesar hati ...
997,TXT0998,@datuakrajoangek Sbaiknya si gemot nga usah ik...
998,TXT0999,kebiasaan merembuk atau bermusyawarah jadi gay...


In [ ]:
!pip install -q nlp_id

In [ ]:
from nlp_id.lemmatizer import Lemmatizer
lemmatizer = Lemmatizer()
lemmatizer.lemmatize('Saya sedang mencoba')

'saya sedang coba'

In [ ]:
train_df['text'] = train_df['text'].apply(remove_noise)
train_df['text'] = train_df['text'].apply(normalize_word)
train_df['text_2'] = train_df['text'].apply(lambda x: lemmatizer.lemmatize(x))
test_df['Text'] = test_df['Text'].apply(remove_noise)
test_df['Text'] = test_df['Text'].apply(normalize_word)
test_df['Text_2'] = test_df['Text'].apply(lambda x: lemmatizer.lemmatize(x))
train_df = drop_duplicated(train_df)
train_df.head()

Df size: (4937, 3)
After dropping empty text rows, df size:  (4929, 3)
After dropping literal duplicated rows, df size:  (4300, 3)
After dropping text duplicated rows, df size:  (4191, 3)


                                                text             label  \
0  Kunjungan Prabowo ini untuk meresmikan dan men...  Sumber Daya Alam   
1  Anies dapat tepuk tangan meriah saat jadi Rekt...           Politik   
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi   
3  Sewaktu anies bersikap kritis ke kinerja pak p...           Politik   
4  Anies Baswedan Harap aparatur sipil negara ter...           Politik   

                                              text_2  
0  kunjung prabowo ini untuk resmi dan serah proy...  
1  anies dapat tepuk tangan riah saat jadi rektor...  
2  emng bener sih dukung 01 ada yang goblok begit...  
3  waktu anies sikap kritis ke kerja pak prabowo ...  
4  anies baswedan harap aparatur sipil negara mas...  

In [ ]:
train_df.label.value_counts()

label
Politik                    2832
Sosial Budaya               382
Ekonomi                     264
Pertahanan dan Keamanan     255
Ideologi                    253
Sumber Daya Alam            131
Demografi                    55
Geografi                     19
Name: count, dtype: int64

In [ ]:
train_df['text'] = train_df['text'].str.lower()
train_df['text_2'] = train_df['text_2'].str.lower()

test_df['Text'] = test_df['Text'].str.lower()
test_df['Text_2'] = test_df['Text_2'].str.lower()

In [ ]:
test_df

      IDText                                               Text  \
0    TXT0001  lu mau orang 2 pro demokrasi di negara ini bis...   
1    TXT0002  prabowo ditanya soal hutang luar negeri dia me...   
2    TXT0003  kiki daliyo ganjar pranowo itulah beliau sosok...   
3    TXT0004  prabowo gibran yang bisa melakukan itu semua d...   
4    TXT0005  lah justru yang gak nyambung junjungan elu aom...   
..       ...                                                ...   
995  TXT0996  bikin bangga deh ganjar mahfud mau alokasikan ...   
996  TXT0997  pak jokowi sebelum pemilihan presiden 2024 ber...   
997  TXT0998  sbaiknya si gemot nga usah ikutan debat deh ng...   
998  TXT0999  kebiasaan merembuk atau bermusyawarah jadi gay...   
999  TXT1000  mirage ditolak juwono dibeli prabowo jubir tim...   

                                                Text_2  
0    lu mau orang 2 pro demokrasi di negara ini bis...  
1    prabowo tanya soal hutang luar negeri dia jawa...  
2    kiki daliyo ganjar 

In [ ]:
# train_df[train_df['text'].isin(test_df['Text'])].drop_duplicates(subset=['text'])

,text,label
64,anies itu prestasinya di jakarta banyak sekali...,Politik
68,isu palestina tidak penting bagi prabowo tidak...,Politik
87,palestina di tindas karena gk punya kekuatan m...,Pertahanan dan Keamanan
156,al fatihah buat alm lambang babar purnomo alum...,Pertahanan dan Keamanan
160,partai politik tanggapi jokowi prabowo makan m...,Politik
...,...,...
4869,jadi ikut sedih juga pak prabowo memang sering...,Politik
4881,saat ini warga kampung bayam seharusnya bisa h...,Politik
4946,percayalah orang pertama yang bakal didepak pr...,Politik
4953,anies kehilangan otoritas bicara etika pada de...,Politik


In [ ]:
kw_df = pd.read_excel(f"{DATA_DIR}/keywords_prob.xlsx")
kw_df.head()

                   keyword  Demografi  Ekonomi  Geografi  Ideologi  \
0        pendidikan rendah        1.0      0.0       0.0       0.0   
1           kelompok etnis        1.0      0.0       0.0       0.0   
2  masyarakat lintas etnis        1.0      0.0       0.0       0.0   
3             kaum difabel        1.0      0.0       0.0       0.0   
4                    bugis        1.0      0.0       0.0       0.0   

   Pertahanan dan Keamanan  Sosial Budaya  Sumber Daya Alam  Politik  
0                      0.0            0.0               0.0      0.0  
1                      0.0            0.0               0.0      0.0  
2                      0.0            0.0               0.0      0.0  
3                      0.0            0.0               0.0      0.0  
4                      0.0            0.0               0.0      0.0  

In [ ]:
test_df

,IDText,Text
0,TXT0001,lu mau orang 2 pro demokrasi di negara ini bis...
1,TXT0002,prabowo ditanya soal hutang luar negeri dia me...
2,TXT0003,kiki daliyo ganjar pranowo itulah beliau sosok...
3,TXT0004,prabowo gibran yang bisa melakukan itu semua d...
4,TXT0005,lah justru yang gak nyambung junjungan elu aom...
...,...,...
995,TXT0996,bikin bangga deh ganjar mahfud mau alokasikan ...
996,TXT0997,pak jokowi sebelum pemilihan presiden 2024 ber...
997,TXT0998,sbaiknya si gemot nga usah ikutan debat deh ng...
998,TXT0999,kebiasaan merembuk atau bermusyawarah jadi gay...


In [ ]:
for kw in kw_df['keyword']:
    train_df[kw] = train_df['text'].str.lower().str.contains(kw).astype(int)
    test_df[kw] = test_df['Text'].str.lower().str.contains(kw).astype(int)

train_df.head()

Streaming output truncated to the last 5000 lines.
<ipython-input-11-a259bd4c0cd7>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[kw] = train_df['text'].str.lower().str.contains(kw).astype(int)
<ipython-input-11-a259bd4c0cd7>:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[kw] = test_df['Text'].str.lower().str.contains(kw).astype(int)
<ipython-input-11-a259bd4c0cd7>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

                                                text             label  \
0  kunjungan prabowo ini untuk meresmikan dan men...  Sumber Daya Alam   
1  anies dapat tepuk tangan meriah saat jadi rekt...           Politik   
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi   
3  sewaktu anies bersikap kritis ke kinerja pak p...           Politik   
4  anies baswedan harap aparatur sipil negara ter...           Politik   

                                              text_2  pendidikan rendah  \
0  kunjung prabowo ini untuk resmi dan serah proy...                  0   
1  anies dapat tepuk tangan riah saat jadi rektor...                  0   
2  emng bener sih dukung 01 ada yang goblok begit...                  1   
3  waktu anies sikap kritis ke kerja pak prabowo ...                  0   
4  anies baswedan harap aparatur sipil negara mas...                  0   

   kelompok etnis  masyarakat lintas etnis  kaum difabel  bugis  \
0               0                    

In [ ]:
# drop rows where all the columns is 0
train_df.drop(train_df[(train_df[kw_df['keyword']] == 0).all(axis=1)].index, inplace=True)
# test_df.drop(test_df[(test_df[kw_df['keyword']] == 0).all(axis=1)].index, inplace=True)

In [ ]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

In [ ]:
class_weights_dict = dict()
n, num_class = train_df.shape[0], len(train_df['label'].unique())
for label in train_df['label'].unique():
    n_l = train_df[train_df['label'] == label].shape[0]
    class_weights_dict[label] = n / (num_class * n_l)

class_weights_dict

{'Sumber Daya Alam': 4.089285714285714,
 'Politik': 0.18534172661870504,
 'Demografi': 9.368181818181819,
 'Pertahanan dan Keamanan': 2.0365612648221343,
 'Ideologi': 2.061,
 'Ekonomi': 1.959125475285171,
 'Sosial Budaya': 1.3667108753315649,
 'Geografi': 28.625}

In [ ]:
train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

text_feat = ["text_2"]

train_pool = Pool(
    data=train_df_split[kw_df["keyword"].to_list() + text_feat].to_numpy(),
    label=train_df_split['label'].to_numpy(),
    text_features=text_feat,
    feature_names=kw_df["keyword"].to_list() + text_feat
)

val_pool = Pool(
    data=val_df_split[kw_df["keyword"].to_list() + text_feat].to_numpy(),
    label=val_df_split['label'].to_numpy(),
    text_features=text_feat,
    feature_names=kw_df["keyword"].to_list() + text_feat
)

In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = class_weights_dict,
    learning_rate=0.01,
    iterations = 4700
)

cb_model.fit(
    train_pool,
    eval_set=val_pool
)

0:	learn: 0.3924045	test: 0.3423711	best: 0.3423711 (0)	total: 57ms	remaining: 4m 27s
100:	learn: 0.5934828	test: 0.5000516	best: 0.5165562 (50)	total: 2.35s	remaining: 1m 46s
200:	learn: 0.6119689	test: 0.5015360	best: 0.5165562 (50)	total: 3.53s	remaining: 1m 19s
300:	learn: 0.6306134	test: 0.5002013	best: 0.5165562 (50)	total: 5.66s	remaining: 1m 22s
400:	learn: 0.6378690	test: 0.5072836	best: 0.5165562 (50)	total: 6.83s	remaining: 1m 13s
500:	learn: 0.6554009	test: 0.5129060	best: 0.5165562 (50)	total: 8.08s	remaining: 1m 7s
600:	learn: 0.6763847	test: 0.5165574	best: 0.5165755 (593)	total: 9.26s	remaining: 1m 3s
700:	learn: 0.6869768	test: 0.5182762	best: 0.5184899 (679)	total: 12.7s	remaining: 1m 12s
800:	learn: 0.6958373	test: 0.5149083	best: 0.5185544 (721)	total: 14.5s	remaining: 1m 10s
900:	learn: 0.7025775	test: 0.5164133	best: 0.5185544 (721)	total: 15.6s	remaining: 1m 5s
1000:	learn: 0.7071919	test: 0.5200736	best: 0.5200736 (997)	total: 16.7s	remaining: 1m 1s
1100:	learn:

In [ ]:
train_df['label'].value_counts()

label
Politik                    2780
Sosial Budaya               377
Ekonomi                     263
Pertahanan dan Keamanan     253
Ideologi                    250
Sumber Daya Alam            126
Demografi                    55
Geografi                     18
Name: count, dtype: int64

In [ ]:
# Only Text
# Make predictions on the validation set
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5366650603440162


In [ ]:
model_1 = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=0,
    random_state=42,
    class_weights=class_weights_dict,
    text_features=['text'],
    learning_rate=0.01,
)


cv_method = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=42)
cv_results_model_1 = cross_val_score(estimator=model_1, X=train_df.drop("label", axis=1), y=train_df["label"], cv=cv_method,
                                     verbose=100, scoring='balanced_accuracy')

print("Model 1 - Balanced Accuracy scores: ")
print(cv_results_model_1)
print("Mean: ")
print(cv_results_model_1.mean().round(3))

In [ ]:
text_feat = ["text"]

train_pool = Pool(
    data=train_df[kw_df["keyword"].to_list() + text_feat].to_numpy(),
    label=train_df['label'].to_numpy(),
    text_features=text_feat,
    feature_names=kw_df["keyword"].to_list() + text_feat
)

In [ ]:
model_1 = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = class_weights_dict,
    learning_rate = 0.015
)

model_1.fit(
    train_pool,
    # eval_set=val_pool
)

0:	learn: 0.4289170	total: 27.4ms	remaining: 27.4s
100:	learn: 0.5996004	total: 1.47s	remaining: 13.1s
200:	learn: 0.6377691	total: 2.62s	remaining: 10.4s
300:	learn: 0.6526425	total: 3.73s	remaining: 8.67s
400:	learn: 0.6664081	total: 4.86s	remaining: 7.26s
500:	learn: 0.6815249	total: 8.89s	remaining: 8.86s
600:	learn: 0.6955730	total: 10s	remaining: 6.64s
700:	learn: 0.7035492	total: 11.1s	remaining: 4.74s
800:	learn: 0.7098427	total: 12.2s	remaining: 3.03s
900:	learn: 0.7204567	total: 13.3s	remaining: 1.46s
999:	learn: 0.7225198	total: 14.4s	remaining: 0us


In [ ]:
train_df

                                                   text  \
0     kunjungan prabowo ini untuk meresmikan dan men...   
1     anies dapat tepuk tangan meriah saat jadi rekt...   
2     emng bener sih pendukung 01 ada yang goblok be...   
3     sewaktu anies bersikap kritis ke kinerja pak p...   
4     anies baswedan harap aparatur sipil negara ter...   
...                                                 ...   
4993  ratusan ribu hektar prabowo menguasai tanah di...   
4995  ngeliat debat kemaren pas prabowo kicep kekira...   
4996  masyarakat yakin bahwa prabowo gibran memiliki...   
4997  imo both are irrational but yang satu jauh leb...   
4998  look at that pak ganjar anda sdh berkecimpung ...   

                        label  pendidikan rendah  kelompok etnis  \
0            Sumber Daya Alam                  0               0   
1                     Politik                  0               0   
2                   Demografi                  1               0   
3                  

In [ ]:
# rename Text to text from test
test_df.rename(columns={"Text": "text", "Text_2": "text_2"}, inplace=True)

In [ ]:
text_feat = ['text']
test_pool = Pool(
    data=test_df[kw_df["keyword"].to_list() + text_feat].to_numpy(),
    # label=test_df['label'].to_numpy(),
    text_features=text_feat,
    feature_names=kw_df["keyword"].to_list() + text_feat
)

In [ ]:
pred = model_1.predict(test_pool)

In [ ]:
pred_2 = model_1.predict(test_pool)

In [ ]:
# flatten pred
pred = [item for sublist in pred for item in sublist]

# check value counts
pd.Series(pred).value_counts()

Politik                    380
Ekonomi                    198
Pertahanan dan Keamanan    140
Ideologi                   134
Sosial Budaya               88
Sumber Daya Alam            47
Demografi                   10
Geografi                     3
Name: count, dtype: int64

In [ ]:
# flatten pred
pred_2 = [item for sublist in pred_2 for item in sublist]

# check value counts
pd.Series(pred_2).value_counts()

Politik                    422
Ekonomi                    179
Pertahanan dan Keamanan    132
Ideologi                   122
Sosial Budaya               84
Sumber Daya Alam            47
Demografi                    9
Geografi                     5
Name: count, dtype: int64

In [ ]:
test_df['label_lemma'] = pred

<ipython-input-29-3c690f4821a8>:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df['label_lemma'] = pred


In [ ]:
test_df['label_polos'] = pred_2

<ipython-input-38-e634feee9dd3>:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df['label_polos'] = pred_2


In [ ]:
test_df.to_csv(f"{DATA_DIR}/pred_catboost.csv", index=False)

In [ ]:
test_df[['IDText',	'text',	'text_2', 'label_lemma', 'label_polos']].to_csv(f"{DATA_DIR}/new_pred.csv", index=False)

In [ ]:
test_df.head()

    IDText                                               text  \
0  TXT0001  lu mau orang 2 pro demokrasi di negara ini bis...   
1  TXT0002  prabowo ditanya soal hutang luar negeri dia me...   
2  TXT0003  kiki daliyo ganjar pranowo itulah beliau sosok...   
3  TXT0004  prabowo gibran yang bisa melakukan itu semua d...   
4  TXT0005  lah justru yang gak nyambung junjungan elu aom...   

                                              text_2  pendidikan rendah  \
0  lu mau orang 2 pro demokrasi di negara ini bis...                  0   
1  prabowo tanya soal hutang luar negeri dia jawa...                  0   
2  kiki daliyo ganjar pranowo itu beliau sosok ya...                  0   
3  prabowo gibran yang bisa laku itu semua demi s...                  0   
4  lah justru yang gak nyambung junjung elu aomkm...                  0   

   kelompok etnis  masyarakat lintas etnis  kaum difabel  bugis  \
0               0                        0             0      0   
1               0       

In [ ]:
train_df['label'].value_counts()

Politik                    2816
Sosial Budaya               379
Ekonomi                     265
Pertahanan dan Keamanan     257
Ideologi                    252
Sumber Daya Alam            128
Demografi                    55
Geografi                     18
Name: label, dtype: int64

In [ ]:
# read pickle to dataframe with_embed_test.pkl
test_df_embed = pd.read_pickle(f"{DATA_DIR}/with_embed_test_full.pkl")
test_df_embed.head()

,IDText,text,text_embed
0,TXT0001,lu mau org 2 pro demokrasi di negara ini bisa ...,"[0.028984811, -0.036268596, -0.02264847, -0.04..."
1,TXT0002,prabowo ditanya soal hutang luar negeri dia me...,"[0.02540058, 0.0014542198, -0.020809272, -0.00..."
2,TXT0003,kiki daliyo ganjar pranowo itulah beliau sosok...,"[0.0037599015, -0.014348123, -0.020956453, -0...."
3,TXT0004,prabowo gibran yang bisa melakukan itu semua d...,"[0.031111086, -0.010258717, -0.010770987, -0.0..."
4,TXT0005,lah justru yang gak nyambung junjungan elu aom...,"[0.0066329995, -0.013745376, -0.0006121621, -0..."


In [ ]:
test_df_embed

,IDText,text,text_embed
0,TXT0001,lu mau org 2 pro demokrasi di negara ini bisa ...,"[0.028984811, -0.036268596, -0.02264847, -0.04..."
1,TXT0002,prabowo ditanya soal hutang luar negeri dia me...,"[0.02540058, 0.0014542198, -0.020809272, -0.00..."
2,TXT0003,kiki daliyo ganjar pranowo itulah beliau sosok...,"[0.0037599015, -0.014348123, -0.020956453, -0...."
3,TXT0004,prabowo gibran yang bisa melakukan itu semua d...,"[0.031111086, -0.010258717, -0.010770987, -0.0..."
4,TXT0005,lah justru yang gak nyambung junjungan elu aom...,"[0.0066329995, -0.013745376, -0.0006121621, -0..."
...,...,...,...
995,TXT0996,bikin bangga deh ganjar mahfud mau alokasikan ...,"[0.023778459, 0.00020238406, -0.0074011376, -0..."
996,TXT0997,pak jokowi sebelum pilpres 2024 berbesar hati ...,"[0.024797702, 0.0023467692, -0.023014598, -0.0..."
997,TXT0998,sbaiknya si gemot nga usah ikutan debat deh ng...,"[0.033157546, -0.0061428966, -0.040251214, -0...."
998,TXT0999,kebiasaan merembuk atau bermusyawarah jadi gay...,"[0.021183468, -0.012528644, -0.012749161, -0.0..."


In [ ]:
train_df_embed = pd.read_pickle(f"{DATA_DIR}/with_embed_full.pkl")
train_df_embed.head()
# reset train_df index

,text,label,text_embed
0,kunjungan prabowo ini untuk meresmikan dan men...,Sumber Daya Alam,"[0.007652878, -0.0030882277, -0.011967206, -0...."
1,anies dapat tepuk tangan meriah saat jadi rekt...,Politik,"[-0.0008003995, -0.017181937, 0.0012290998, -0..."
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi,"[0.021622209, -0.010625936, -0.015348185, -0.0..."
3,sewaktu anies bersikap kritis ke kinerja pak p...,Politik,"[0.02223998, -0.024771843, -0.012059052, -0.02..."
4,anies baswedan harap aparatur sipil negara ter...,Politik,"[0.020082695, -0.009796472, -0.018930588, -0.0..."


In [ ]:
train_df_embed['text'] = train_df_embed['text'].apply(remove_noise)
train_df_embed['text'] = train_df_embed['text'].apply(normalize_word)
train_df_embed = drop_duplicated(train_df_embed)
train_df_embed.reset_index(drop=True, inplace=True)
train_df_embed.head()

Df size: (5000, 3)
After dropping empty text rows, df size:  (4991, 3)
After dropping literal duplicated rows, df size:  (4348, 3)
After dropping text duplicated rows, df size:  (4235, 3)


,text,label,text_embed
0,kunjungan prabowo ini untuk meresmikan dan men...,Sumber Daya Alam,"[0.007652878, -0.0030882277, -0.011967206, -0...."
1,anies dapat tepuk tangan meriah saat jadi rekt...,Politik,"[-0.0008003995, -0.017181937, 0.0012290998, -0..."
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi,"[0.021622209, -0.010625936, -0.015348185, -0.0..."
3,sewaktu anies bersikap kritis ke kinerja pak p...,Politik,"[0.02223998, -0.024771843, -0.012059052, -0.02..."
4,anies baswedan harap aparatur sipil negara ter...,Politik,"[0.020082695, -0.009796472, -0.018930588, -0.0..."


In [ ]:
# get all index in test that has consine similiarity > 0.95 with train, calculat cosine similiarity using text_embed column

from sklearn.metrics.pairwise import cosine_similarity

for i, row in tqdm(test_df_embed.iterrows(), total=test_df_embed.shape[0]):
    test_df_embed.loc[i, "cosine_sim"] = cosine_similarity(
        [row["text_embed"]],
        train_df_embed["text_embed"].to_list()
    ).max()

In [ ]:
test_df_embed[test_df_embed['cosine_sim'] > 0.98]

,IDText,text,text_embed,cosine_sim
11,TXT0012,malu maluin dari gemoy ke minyak goreng apakah...,"[0.026865698, -0.03133889, -0.008080593, -0.03...",1.000000
20,TXT0021,upaya pemberantasan korupsi anies ingin kembal...,"[0.011423329, 0.0014343983, -0.009302331, -0.0...",1.000000
22,TXT0023,capres cawapres ganjar pranowo dan mahfud mari...,"[0.01055791, -0.021979617, -0.016377067, -0.03...",1.000000
25,TXT0026,abah online meroket figur pak anies semakin po...,"[0.021367706, -0.007478056, -0.008939597, -0.0...",1.000000
45,TXT0046,program pasangan capres cawapres ganjar pranow...,"[0.016749667, -0.022329586, -0.019494614, -0.0...",0.984032
...,...,...,...,...
969,TXT0970,survei media ekonomi bisnis terkemuka asal as ...,"[0.0212464, -0.013048108, -0.012299449, -0.016...",1.000000
972,TXT0973,pernyataan prabowo ttg gaza dilindas karena le...,"[0.032448944, 0.017916357, -0.0016202913, -0.0...",1.000000
985,TXT0986,menjegal videotron anies baswedan hanya bagian...,"[0.02434459, -0.0053591244, -0.031297565, -0.0...",1.000000
990,TXT0991,percayalah orang pertama yang bakal didepak pr...,"[0.017740678, 0.001892904, -0.021736354, -0.04...",1.000000


In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Assuming test_df_embed and train_df_embed are your dataframes

# Convert the text embeddings to list
train_embeds = train_df_embed["text_embed"].to_list()

# Initialize the columns to store the results
test_df_embed["cosine_sim"] = 0.0
test_df_embed["train_text"] = ""
test_df_embed["train_label"] = ""

# Iterate over each row in the test dataframe
for i, row in tqdm(test_df_embed.iterrows(), total=test_df_embed.shape[0]):
    # Calculate cosine similarity with all train embeddings
    similarities = cosine_similarity([row["text_embed"]], train_embeds)

    # Find the maximum similarity and its index
    max_sim = similarities.max()
    max_sim_idx = similarities.argmax()

    # Update the dataframe with the maximum similarity
    test_df_embed.at[i, "cosine_sim"] = max_sim

    # If the similarity is above 0.98, also store the text and label from the train data
    if max_sim > 0:
        test_df_embed.at[i, "train_text"] = train_df_embed.at[max_sim_idx, "text"]
        test_df_embed.at[i, "train_label"] = train_df_embed.at[max_sim_idx, "label"]

In [ ]:
similar_df = test_df_embed[test_df_embed["train_text"] != ""]

In [ ]:
similar_df.to_csv(f"{DATA_DIR}/mirip_all.csv", index=False)

In [ ]:
train_df_embed['label'].value_counts()

Politik                    2866
Sosial Budaya               384
Ekonomi                     264
Pertahanan dan Keamanan     259
Ideologi                    255
Sumber Daya Alam            133
Demografi                    55
Geografi                     19
Name: label, dtype: int64

In [ ]:
# loop trough test_df_embed and test_df
for i, row in similar_df.iterrows():
    # print test_df at index i
    if similar_df.loc[i, 'train_label'] == 'Politik' and similar_df.loc[i, 'cosine_sim'] >= 0.98:
        test_df.loc[i,'label'] = 'Politik'
    # print(test_df.loc[i]['label'])
    elif similar_df.loc[i, 'train_label'] == 'Ekonomi' and similar_df.loc[i, 'cosine_sim'] >= 0.95:
        test_df.loc[i,'label'] = 'Ekonomi'
    elif similar_df.loc[i, 'train_label'] == 'Geografi' and similar_df.loc[i, 'cosine_sim'] >= 0.909849882125854:
        test_df.loc[i,'label'] = 'Geografi'
    elif similar_df.loc[i, 'train_label'] == 'Ideologi' and similar_df.loc[i, 'cosine_sim'] >= 0.938026428222656:
        test_df.loc[i,'label'] = 'Ideologi'
    elif similar_df.loc[i, 'train_label'] == 'Pertahanan dan Keamanan' and similar_df.loc[i, 'cosine_sim'] >= 0.952700495719909:
        test_df.loc[i,'label'] = 'Pertahanan dan Keamanan'
    elif similar_df.loc[i, 'train_label'] == 'Sosial Budaya' and similar_df.loc[i, 'cosine_sim'] >= 0.942158102989196:
        test_df.loc[i,'label'] = 'Sosial Budaya'

In [ ]:
test_df['label'].value_counts()

Politik                    463
Ekonomi                    154
Ideologi                   139
Pertahanan dan Keamanan    129
Sosial Budaya               87
Sumber Daya Alam            13
Demografi                    9
Geografi                     6
Name: label, dtype: int64

In [ ]:
leak = pd.read_csv(f'{DATA_DIR}/test_leakage.csv')

In [ ]:
leak

,IDText,label_dup
0,TXT0001,NaN
1,TXT0002,NaN
2,TXT0003,NaN
3,TXT0004,NaN
4,TXT0005,NaN
...,...,...
995,TXT0996,NaN
996,TXT0997,NaN
997,TXT0998,NaN
998,TXT0999,NaN


In [ ]:
for i, row in leak.iterrows():
    # check if nan
    if leak.loc[i, 'label_dup'] is not np.nan:
        test_df.loc[i, 'label'] = leak.loc[i, 'label_dup']

In [ ]:
test_df['label'].value_counts()

Politik                    463
Ekonomi                    154
Ideologi                   139
Pertahanan dan Keamanan    130
Sosial Budaya               86
Sumber Daya Alam            13
Demografi                    9
Geografi                     6
Name: label, dtype: int64

In [ ]:
submission_gaming = pd.DataFrame()

In [ ]:
submission_gaming[['IDText', 'Kelas']] = test_df[['IDText', 'label']]

In [ ]:
submission_gaming.to_csv(f'{DATA_DIR}/submission_gaming.csv', index=False)